# Parallel dataset generation — multicore recordings

Same model / config as **`neuron_network_simulation.ipynb`** (one fixed seed-1
network; normal vs seizure differ only by `sahp_ainc_slow`), but the recordings
run **concurrently across CPU cores** instead of one-at-a-time.

Each recording runs as a *separate OS process* (robust on Windows + Jupyter; fresh
NEURON state per process). All recordings share the same fixed wiring and differ
only by their per-recording noise reseed `rec_idx` — so each recording is a
genuinely different trial, and the output is **equivalent to
`workflows.generate_dataset`, just faster**.

A **memory-aware worker cap** keeps peak RAM ≈ (workers × `PER_WORKER_GB`) under
the currently-free RAM, so parallelism won't OOM the machine.

> Uses `neuron_simulation.parallel_dataset`. Requires compiled mechanisms
> (`nrnivmodl mechanisms` in `neuron_simulation/`), same as the other notebooks.

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

from neuron_simulation import parallel_dataset as pds, states
from neuron_simulation.topology import NeuronWeightParameters

# ==========================================================================
# CONFIG — identical to notebooks/neuron_network_simulation.ipynb.
# ONE fixed network (seed 1). Normal vs seizure differ by a SINGLE parameter:
#   sahp_ainc_slow (slow-AHP / M-current strength).
# ==========================================================================
wp = NeuronWeightParameters()
wp.within_exc_range  = (0.0010, 0.0022)   # exc weight, same cluster
wp.between_exc_range = (0.0008, 0.0016)   # exc weight, across clusters
wp.within_inh_range  = (0.0025, 0.0055)   # inh weight, same cluster
wp.between_inh_range = (0.0020, 0.0040)   # inh weight, across clusters
wp.use_lognormal     = True
wp.lognormal_sigma   = 0.5

SAHP_NORMAL, SAHP_SEIZURE = 0.01, 0.004   # the ONLY thing that differs between states

CONFIG = {
    'topology': dict(
        num_clusters=50, neurons_per_cluster_range=(4, 40),
        inhibitory_probability=0.2, cluster_radius=1.0, space_size=15.0, seed=1,
        decay_sigma=3.0, max_connection_distance=6.0,
        cell_type_specific=True,
        p_ee_within=0.2, p_ee_between=0.1, p_ei_within=0.20, p_ei_between=0.08,
        p_ie_within=0.40, p_ii_within=0.50,
        within_cluster_prob=0.25, between_cluster_prob=0.06,
        ln_sigma=0.5, target_density=None, weight_params=wp,
    ),
    'build': dict(
        synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
        exc_weight_scale=2.0, inh_weight_scale=2.5, depression_d=0.2, tau_d=500.0,
        noise_rate=5.0, noise_weight=0.004, adapt=True,
        sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
        sahp_ainc_slow=SAHP_NORMAL, sahp_tau_slow=6500.0, delay_per_distance=2.0,
    ),
    'sim': dict(dt=0.05, discard_transient_ms=1000.0),
}

def build_for(sahp_slow):
    """build_kwargs for one state: identical network, only sahp_ainc_slow differs."""
    b = dict(CONFIG['build']); b['sahp_ainc_slow'] = sahp_slow; return b

# ---- dataset settings (match neuron_network_simulation) ----
N_RECORDINGS   = 15
RECORDING_MS   = 60000.0
RECORD_VOLTAGE = True          # matches the notebook -- NOTE: drives per-worker memory UP
SAVE_RASTERS   = True

# ---- parallelism / memory ----
MAX_WORKERS    = None          # None -> auto (min of cpu, n_recordings, memory cap)
PER_WORKER_GB  = 1.0           # ~926 cells + 60s voltage ~= 0.6-0.8 GB; 1.0 leaves margin
HEADROOM_GB    = 4.0           # RAM to leave free for the OS / other apps

print('config ready (matches neuron_network_simulation): SAHP_NORMAL=%.3f SAHP_SEIZURE=%.3f | '
      '%d x %.0fs | record_voltage=%s' % (SAHP_NORMAL, SAHP_SEIZURE, N_RECORDINGS, RECORDING_MS/1000, RECORD_VOLTAGE))

## 1. Preview the worker count (memory-aware, no simulation yet)

Reads current free RAM and picks a safe number of concurrent workers so that
peak memory ≈ workers × `PER_WORKER_GB` stays under free RAM minus `HEADROOM_GB`.
With `RECORD_VOLTAGE=True` on the ~926-cell network the footprint is larger, so
this will (correctly) choose fewer workers when RAM is tight.

In [ ]:
workers, free_gb, cpu = pds.pick_worker_count(
    N_RECORDINGS, per_worker_gb=PER_WORKER_GB, headroom_gb=HEADROOM_GB, max_workers=MAX_WORKERS)
print('CPU cores        : %d' % cpu)
print('Free RAM         : %.1f GB' % free_gb)
print('Per-worker (est) : %.2f GB   headroom %.0f GB' % (PER_WORKER_GB, HEADROOM_GB))
print('-> will use      : %d concurrent workers' % workers)
print('   added RAM est : ~%.1f GB peak' % (workers * PER_WORKER_GB))

## 2. Generate both datasets in parallel

Mirrors `neuron_network_simulation` cell 12: two datasets from the SAME wired
network (seed 1 → identical ground truth), differing only by `sahp_ainc_slow`
(normal vs seizure). Each writes the usual session bundle (`network_*.npz`,
`recording###.npz`, `session_metadata.json`) plus per-recording `.log` files.

> Saved under `NEURON data parallel/` so it does not collide with the sequential
> notebook's `NEURON data/`.

In [ ]:
meta_n, dir_n = pds.generate_dataset_parallel(
    n_recordings=N_RECORDINGS, recording_duration=RECORDING_MS,
    topology_kwargs=CONFIG['topology'], build_kwargs=build_for(SAHP_NORMAL),
    state=states.normal_state(), record_voltage=RECORD_VOLTAGE, save_rasters=SAVE_RASTERS,
    save_dir='NEURON data parallel/normal',
    max_workers=MAX_WORKERS, per_worker_gb=PER_WORKER_GB, headroom_gb=HEADROOM_GB,
    **CONFIG['sim'])
print('normal dataset  ->', dir_n)

meta_s, dir_s = pds.generate_dataset_parallel(
    n_recordings=N_RECORDINGS, recording_duration=RECORDING_MS,
    topology_kwargs=CONFIG['topology'], build_kwargs=build_for(SAHP_SEIZURE),
    state=states.normal_state(), record_voltage=RECORD_VOLTAGE, save_rasters=SAVE_RASTERS,
    save_dir='NEURON data parallel/seizure',
    max_workers=MAX_WORKERS, per_worker_gb=PER_WORKER_GB, headroom_gb=HEADROOM_GB,
    **CONFIG['sim'])
print('seizure dataset ->', dir_s)

## 3. Verify — recordings are each different

Same fixed wiring, different noise per recording → spike counts differ across
recordings (they are NOT identical copies).

In [ ]:
print('normal  per-rec spikes:', [r.get('num_spikes') for r in meta_n['recordings']])
print('seizure per-rec spikes:', [r.get('num_spikes') for r in meta_s['recordings']])
for name, m in [('normal', meta_n), ('seizure', meta_s)]:
    ok = sum(1 for r in m['recordings'] if r.get('success'))
    counts = [r.get('num_spikes') for r in m['recordings'] if r.get('success')]
    print('%-8s: %d/%d ok | %d distinct spike-counts -> %s' % (
        name, ok, len(m['recordings']), len(set(counts)),
        'each recording DIFFERENT' if len(set(counts)) > 1 else 'identical (unexpected!)'))

## Notes

- **Equivalent to sequential.** Recording *k* here == recording *k* from
  `workflows.generate_dataset` with the same config — noise is
  `Random123(base_seed, gid, rec_idx)`, deterministic regardless of order/process.
  Parallelism changes *wall time*, not the data.
- **Speedup** ≈ `min(workers, N)`×.
- **Memory / OOM.** Each worker holds ONE recording at a time and frees it after
  saving, so peak ≈ workers × `PER_WORKER_GB`, capped against *currently-free* RAM.
  `RECORD_VOLTAGE=True` (as here) makes each worker large — a 60 s per-neuron
  voltage trace on ~926 cells is several hundred MB — so the cap will pick fewer
  workers when RAM is tight. Set `RECORD_VOLTAGE=False` to run many more workers.
- **Different *networks* per recording?** This shares one wiring (for inference on
  a fixed graph). To vary the graph too, change `topology['seed']` per recording —
  ask and I'll add that mode.